---
title: PLACEHOLDER
subtitle: Alignment Summary
date: "9999-12-31"
edit_url: null
---

## Linked-Read Metrics
**Linked-read type**: PLACEHOLDER

In [ ]:
import altair as alt
from harpy.report.components import colored_boxes, print_html, standard_itable
from harpy.report.utilities import binned_histogram, nxx, last_line
from harpy.report.theme import palette
import itables
import os
import numpy as np
import pandas as pd
import sys
itables.init_notebook_mode(connected=True)

In [ ]:
samplename = "sample"
platform = 'haplotagging'
mol_dist = 75000
windowsize = 50000
contigs = "default"

In [ ]:
coverage = f"reports/data/coverage/{samplename}.cov.gz"
molcov = f"reports/data/coverage/{samplename}.molcov.gz"
bxstats = f"reports/data/bxstats/{samplename}.bxstats.gz"
SKIPLR = platform == "none"

In [ ]:
def consolidated_hist(data: pd.Series, linkmask: pd.Series, binwidth: int|float, normalize: bool) -> pd.DataFrame:
    '''
    Invoke `binned_histogram()` once for all the data (a column) and linked-read only data, returning a single DataFrame
    The `linkmask` is a boolean list/series that subsets `data`. Both `nmol` and `nlinkmol` are necessary to
    correctly calculate proportions. This is a shortcut method for not writing the process out in full each time.
    '''
    binned_df = binned_histogram(data, binwidth, normalize).rename(columns = {"proportion": "all reads"})
    binned_dfl = binned_histogram(data[linkmask], binwidth, normalize)
    binned_df['linked reads'] = binned_dfl['proportion']
    if isinstance(binwidth, int):
        binned_df['range_label'] = [f"{i}-{int(i)+binwidth-1}" for i in binned_df['bin'].values.tolist()]
    else:
        digits = len(str(binwidth).split(".")[1])
        binned_df['range_label'] = [f"{i}-{round(float(i)+binwidth,digits)}" for i in binned_df['bin'].values.tolist()]
    return binned_df

def density_plot(data: pd.DataFrame, title:str, samplename:str, filename:str, extra_cols: list[str] = []):
    selection = alt.selection_point(fields=['key'], bind='legend')
    cols = ['all reads','linked reads'] + extra_cols
    return (alt.Chart(data)
        .transform_fold(cols)
        .transform_filter(selection)
        .mark_area(interpolate = "monotone")
        .add_params(selection)
        .configure_legend(orient = "top")
        .properties(
            title=alt.Title(title, subtitle = samplename),
            usermeta={'embedOptions': {'downloadFileName': f'{samplename}.{filename}'}}
        )
    )

In [ ]:
if not SKIPLR:
    tb = pd.read_table(bxstats, sep = '\t').drop(columns = ["start", "end"])
    if len(tb) == 0:
        SKIPLR = True
        print_html(f"Input data file {bxstats} is empty")
    else:
        valids = tb.iloc[np.where(tb.molecule.values > 0)]
        invalids = tb.iloc[np.where(tb.molecule.values < 0)]
        n_mol = len(valids.index)
        SKIPLR = SKIPLR  | (n_mol == 0)

In [ ]:
if SKIPLR:
    print_html(f"reports/data/bxstats/{os.path.basename(bxstats)} has no valid barcodes, skipping linked-read metrics.")
else:
    nBX = valids.groupby('contig').size().reset_index(name='nBX')
    avgBX = round(nBX['nBX'].mean())
    totuniqBX = int(last_line(bxstats).split(":")[-1])
    tot_valid = valids['reads'].sum()
    tot_invalid = invalids['reads'].sum()
    n_reads = tb['reads'].sum()
    perc_valid = round(tot_valid / n_reads, 4)

    mask = valids['reads'] >= 2
    n_linked_reads = valids.loc[mask, 'reads'].sum()
    n_linked_mol = mask.sum() # sum of Trues

    linked_perc_abs = round(n_linked_reads / n_reads, 4)
    linked_perc = round(n_linked_reads / tot_valid, 4)
    linked_perc_mol = round(n_linked_mol / n_mol, 4)
    boxes = colored_boxes()
    boxes.add(len(nBX.index), "Contigs")
    boxes.add(totuniqBX, "Unique Barcodes", width = 215)
    boxes.add(f"{int(mol_dist/1000)}kb" if mol_dist > 0 else "-", "Molecule Thresh.", width = 215)
    boxes.add(valids['molecule'].nunique(), "Unique Molecules")
    boxes.conditional(perc_valid, "Valid BX Recs", 0.5, as_percent = True)
    if n_mol > 0:
        boxes.conditional(linked_perc, "Linked (Relative)", 40, as_percent = True)
        boxes.conditional(linked_perc_abs, "Linked (Absolute)", 40, as_percent = True)
        boxes.add(avgBX, "Avg mol/contig", width = 215)
    else:
        boxes.conditional(0, "Linked",  1, as_percent = True)
        boxes.conditional(0, "Avg mol/contig",1, width = 215)
    boxes.render()

The boxes above show general linked-read library performance metrics[^1].
The table below is a per-contig assessment of the number of valid/invalid (barcoded) alignment records and the molecule NXX stats per contig.

[^1]:
    Linked (relative)
    : refers to the percent of linked records (2 or more single/paired end reads) relative to alignment records with valid linked-read barcodes
    
    Linked (absolute)
    : refers to the percent of linked records out of all the records for this sample, including those with invalid or absent linked-read barcodes
    
    These values should be the same if there are no barcode-invalid records

In [ ]:
if SKIPLR:
    print_html(f"Skipped")

else:
    # Create summary_table
    summary_table = valids.groupby('contig').agg(
        valid_records=('reads', 'sum'),
        molecules=('molecule', 'size'),  # 'size' counts the number of rows
        n50=('length_inferred', lambda x: nxx(x, 50)),
        n75=('length_inferred', lambda x: nxx(x, 75)),
        n90=('length_inferred', lambda x: nxx(x, 90))
    ).reset_index()

    # Create invalids summary
    summ_invalid = invalids.groupby('contig').agg(
        invalid_records=('reads', 'sum')
    ).reset_index()

    # Left join
    summary_table = summary_table.merge(summ_invalid, on='contig', how='left')

    # Reorder columns (contig, valid_records, invalid_records, molecules, n50, n75, n90)
    summary_table = summary_table[['contig', 'valid_records', 'invalid_records', 'molecules', 'n50', 'n75', 'n90']]

    # Replace NaN with 0
    summary_table['invalid_records'] = summary_table['invalid_records'].fillna(0)
    summary_table.columns = ['Contig', 'Valid Records', 'Invalid Records','Molecules', 'N50', 'N75', 'N90']
    standard_itable(summary_table, f"{samplename}_molecule_NX", fixedcols=1)

### Reads per Molecule
This chart shows the distribution of the number of reads per linked molecule. That is, how many alignments are associated with a unique molecule. This excludes the number of reads associated with invalid or absent linked-read barcodes. The `linked` histogram is the percentages recalculated when disregarding singletons.

In [ ]:
if SKIPLR:
    print_html(f"Skipped")
else:

    binned_counts = np.bincount(valids['reads'].astype(int))
    all_perc = (binned_counts / n_reads)
    linked_perc = (binned_counts / n_linked_reads)
    linked_perc[1] = 0
    binned_df = pd.DataFrame({
        'readsper': range(len(binned_counts)),
        'all reads': all_perc,
        'linked reads': linked_perc
    })
    (
        density_plot(binned_df, 'Reads Per Molecule', samplename, 'readspmol')
        .encode(
            x=alt.X('readsper:O', title=None).axis(labelAngle=0)
                .scale(domainMax= binned_df['readsper'].max()),
            y=alt.Y('value:Q', title='Percent of Reads')
                .axis(format='%')
                .stack(False),
            color=alt.Color("key:N").scale(domain = ["all reads", "linked reads"]),
            tooltip = [
                alt.Tooltip("key:N", title = "Data Type"),
                alt.Tooltip("readsper:O", title="Reads Per Molecule"),
                alt.Tooltip("value:Q", format = ".2%", title="% Reads")
            ]
        )
    ).display()

### Bases Aligned Per Molecule
This is a frequency distribution showing the number of base pairs aligned per unique molecule. These data are shown in 500 bp bins. There are two distributions: one with `all reads` (yellow) and another with only `linked reads` (blue). The percent of molecules in the `linked reads` distribution is relative to the total number of linked molecules.

In [ ]:
if SKIPLR:
    print_html(f"Skipped")
else:
    binned_df = consolidated_hist(valids['aligned_bp'], mask, 500, True)
    (
        density_plot(binned_df, 'Bases Aligned Per Molecule', samplename, 'basesper')
        .encode(
            x=alt.X('bin:Q', title="Base Pairs (bp)").axis(tickMinStep=500),
            y=alt.Y('value:Q', title='Percent of Reads')
                .axis(format='%')
                .scale(domain = [0,1])
                .stack(False),
            color=alt.Color("key:N").scale(domain = ["all reads", "linked reads"]),
            tooltip = [
                alt.Tooltip("key:N", title = "Data Type"),
                alt.Tooltip("range_label:N", title="Based Aligned Per Molecule"),
                alt.Tooltip("value:Q", format = ".2%", title="% Molecules")
            ]
        )
    ).display()

### Inferred Molecule Length
This is the frequency distribution of molecule lengths inferred from the first and last alignment positions along a contig for all alignments associated with a single linked-read barcode on a given contig.There are two distributions: one with `all` the alignments (yellow) and another with only `linked reads` (blue). The percent of molecules in the `linked reads` distribution is relative to the total number of linked molecules.

In [ ]:
if SKIPLR:
    print_html(f"Skipped")
else:
    binned_df = consolidated_hist(valids['length_inferred']/1000, mask, 10, True)
    binned_df['bin'] = binned_df['bin'].cat.rename_categories({'0': '1'})
    (
        density_plot(binned_df, 'Inferred Molecule Length', samplename, 'inferredlen')
        .transform_calculate(bin_kb='datum.bin / 1000')
        .encode(
            x=alt.X('bin_kb:Q', title="Kilobases (kbp)").scale(type='log').axis(labelAngle=-40),
            y=alt.Y('value:Q', title='Percent of Molecules')
                .axis(format='%')
                .stack(False),
            color=alt.Color("key:N").scale(domain = ["all reads", "linked reads"]),
            tooltip = [
                alt.Tooltip("key:N", title = "Data Type"),
                alt.Tooltip("range_label:N", title="Kilobases"),
                alt.Tooltip("value:Q", format = ".2%", title="% Molecules")
            ]
        )
    ).display()

### Molecule Read Coverage
This chart shows molecule coverage breadth as computed by the number of bases aligned to that molecule. In other words, "*how much of each unique long molecule is actually sequenced/aligned?*" Keep in mind there is a distinction between sequences and alignments, since some sequences belonging to a particular molecule may not align well and wouldn't appear in the alignment data. This chart also shows molecule coverage breadth as computed by the *inferred fragment* from which the aligned sequences originate. Unlike read coverage, this calculation takes into account the total insert size, including unsequenced gaps between the forward and reverse reads.

In [ ]:
if SKIPLR:
    print_html(f"Skipped")
else:
    binned_df = consolidated_hist(valids['coverage_bp'], mask, 0.05, True)
    binned_df_inf = consolidated_hist(valids['coverage_inserts'], mask, 0.05, True)
    binned_df['all reads (inferred)'] = binned_df_inf['all reads']
    binned_df['linked reads (inferred)'] = binned_df_inf['linked reads']
    (
        density_plot(binned_df, 'Molecule Coverage Breadth', samplename, 'molcov', ['all reads (inferred)','linked reads (inferred)'])
        .encode(
            x=alt.X('bin:Q', title="% Molecule Covered").axis(format = '%'),
            y=alt.Y('value:Q', title='Percent of Molecules')
                .axis(format='%')
                .stack(False),
            color=alt.Color("key:N").scale(domain = ["all reads", "linked reads", "all reads (inferred)", "linked reads (inferred)"]),
            tooltip = [
                alt.Tooltip("key:N", title = "Data Type"),
                alt.Tooltip("range_label:N", title="% Molecule Covered"),
                alt.Tooltip("value:Q", format = ".2%", title="% Molecules")
            ]
        )
    ).display()

---
## Depth Metrics

In [ ]:
# clear out objects in memory to reduce RAM usage
del valids
del invalids
del tb

In [ ]:
tb = pd.read_table(coverage, sep = ' ', header = None, usecols = [0,1,2,5])
_terminate = []
if len(tb) == 0:
    _terminate.append(f"&emsp;- input data file reports/data/coverage/{os.path.basename(coverage)} is empty")

tbmol = pd.read_table(molcov, sep = '\t', header = None)
if len(tbmol) == 0:
    _terminate.append(f"&emsp;- input data file reports/data/coverage/{os.path.basename(molcov)} is empty")

if _terminate:
    print_html("⚠️ Cannot proceed with alignment depth calculations", *_terminate)
    sys.exit(0)

tb.columns = ["Contig", "Position", "Position End", "Read Depth"]
tbmol.columns = ["Contig", "Position", "Position End", "Molecule Depth"]
tb = tb.merge(tbmol, on= ["Contig", "Position", "Position End"])

del tbmol

tb['Read Depth'] = tb['Read Depth'].round(2)
tb['Molecule Depth'] = tb['Molecule Depth'].round(2)

q99 = tb['Read Depth'].quantile(0.99)
mol_q99 = tb['Molecule Depth'].quantile(0.99)
global_avg = round(tb['Read Depth'].mean(),2)
global_sd = tb['Read Depth'].std()
mol_global_avg = round(tb['Molecule Depth'].mean(),2)
mol_global_sd = tb['Molecule Depth'].std()

outliers = tb[tb['Read Depth'] > q99]
nonoutliers = tb[tb['Read Depth'] <= q99]

# group by and summarize
contig_avg = tb.groupby('Contig').agg(
    average=('Read Depth', 'mean'),
    mol_average=('Molecule Depth', 'mean'),
    sdv=('Read Depth', 'std'),
    mol_sdv=('Molecule Depth', 'std')
).reset_index()
contig_avg.columns = ['Contig', 'Average', 'Std Dev', 'Molecule Average', 'Molecule Std Dev']

# add global row
global_row = pd.DataFrame({
    'Contig': ['global'],
    'Average': [global_avg],
    'Std Dev': [global_sd],
    'Molecule Average': [mol_global_avg],
    'Molecule Std Dev': [mol_global_sd]
})

contig_avg = pd.concat([global_row, contig_avg], ignore_index=True)
contig_avg[['Average', 'Std Dev', 'Molecule Average', 'Molecule Std Dev']] = contig_avg[['Average', 'Std Dev', 'Molecule Average', 'Molecule Std Dev']].round(2)

In [ ]:
(
  colored_boxes()
  .add(len(contig_avg) - 1, "Contigs")
  .add(f"{round(windowsize/1000)}kb", "Intervals")
  .add(global_avg, "Avg Depth")
  .add(global_sd, "Stdev Depth")
  .add(mol_global_avg, "Avg Linked Depth")
  .add(mol_global_sd, "Stdev Linked Depth", width = 215)
  .add(q99, "Q99 Alignments")
  .add(mol_q99, "Q99 Inferred Mol")
).render()


### Alignment Depth Distribution

These are the frequencies of interval coverage across all intervals for all contigs. For visual clarity, the distributions are truncated at the 99% quantiles. If the linked depth seems suspiciously high, then it's possible (or likely) that you have distant
alignments sharing barcodes but don't actually originate from the same DNA molecule and instead share the barcode by chance. These
are referred to by a handful of names like "barcode clashing" and can be remedied by deconvolution methods at the pre-
or post-alignment stages. Additionally, many downstream linked-read softwares have internal deconvolution methods to account for 
this, so it might not be something that needs addressing at this stage.

In [ ]:
def depth_hist(col: pd.Series, cutoff: int, precision: int = 3) -> pd.DataFrame:
    '''Convenience function to make a histogram when depth is really low''' 
    data = col[col <= cutoff]
    binned = pd.cut(data * 1000, bins = 30, retbins=True, include_lowest=True)
    binned_counts = binned[0].value_counts(normalize = True).sort_index()
    if precision:
        bins = (binned[1][:-1] / 1000).round(precision)
    else:
        bins = (binned[1][:-1] / 1000).astype(int)
    bins[0] = 0
    labels = []
    for i in range(len(bins)):
        try:
            labels.append(f"{bins[i]}-{bins[i+1]}")
        except IndexError:
            labels.append(f"{bins[i]}-{cutoff}")

    return pd.DataFrame({
        'Bin': bins,
        'Interval': labels,
        'Proportion': (binned_counts.values).round(3)
    })

In [ ]:
readdep = depth_hist(tb['Read Depth'], q99, 3)
readdep['Percent of Intervals'] = 'Read Depth'
moldep = depth_hist(tb['Molecule Depth'], mol_q99, 0)
moldep['Percent of Intervals'] = 'Molecule Depth'
_hist = pd.concat([readdep, moldep])
_hist= _hist[_hist['Proportion'] != 0]

(
    alt.Chart(_hist)        
    .mark_area(interpolate = "basis")
    .encode(
        x=alt.X('Bin:Q')
            .scale(domainMin=0)
            .axis(alt.Axis(title='Depth')),
        y = alt.Y('Proportion:Q', title = None).axis(format = '%'),
        color = alt.Color('Percent of Intervals:N').legend(None),
        row = alt.Row("Percent of Intervals:N").header(labelOrient = 'right'),
        tooltip = [
            alt.Tooltip("Metric:N", title = "Metric"),
            alt.Tooltip('Interval:N', title = "Depth Interval"),
            alt.Tooltip('Proportion:Q', title = "Percent of Intervals").format('.2%')
        ]
    )
    .resolve_scale(x='independent', y='independent')
    .properties(
        height = 150, width = 650,
        usermeta={'embedOptions': {'downloadFileName': f'{samplename}.depth.hist'}}
    )
)

### Depth Metrics
::::{tab-set}
:::{tab-item} Average Depth
This table[^tb] shows the global and per-contig average depth and standard deviation per intervalm **including** intervals whose depth is flagged an outlier in the data.

![](#depthtable)
:::
:::{tab-item} Depth Outliers
This table[^outliers] shows the genomic intervals considered outliers, as determined by having depth greater than the 99% quantile of aligment depths.

![](#depthoutliers)
:::
::::

[^tb]:
    | Column | Description |
    |:-------|:------------|
    | `Contig` | name of the contig|
    | `Average` | average alignment depth|
    | `Std Dev` | standard deviation of alignment depth|
    | `Molecule Average` | average depth including gaps between linked reads|
    | `Molecule Std Dev` | standard deviation of the depth including gaps between linked reads|

[^outliers]:
    | Column | Description |
    |:-------|:------------|
    | `Contig` | name of the contig |
    | `Position` | start of the genomic interval on the contig |
    | `Position End` | end of the genomic interval on the contig |
    | `Read Depth` | number of alignments at the interval |
    | `Molecule Depth` | number of alignments at the interval, including gaps between linked reads |

In [ ]:
#| label: depthtable
standard_itable(contig_avg, f"{samplename}.align.depth", None, 1)

In [ ]:
#| label: depthoutliers
standard_itable(outliers, f"{samplename}.align.outliers", None, 1)

### Depth and Coverage, Visualized
This is a vizualization of depth information across up to 30 of the largest contigs (unless specific contigs were provided).

Read Depth
: These reads are considered to have *proper* alignment in the interval, where "proper" refers to a read not marked as a duplicate or flagged with the SAM `UNMAP`, `SECONDARY`, or `QCFAIL` flags.

Molecule Depth
: The number of *molecules*, i.e. the **linked depth**, which is the alignment depth of the molecules inferred from linked-read data, included unsequenced segments between reads sharing the same linked-read barcode[^ld].

[^ld]:
    It's common for the linked depth histogram to gradually increase towards the center of a contig due to the likelihood of linked molecules spanning the center of a contig.

In [ ]:
if contigs == "default":
    _keep = contigs = tb.groupby('Contig')['Position'].max().nlargest(30).index.tolist()
    tb = tb[tb['Contig'].isin(_keep)].reset_index()
else:
    tb = tb[tb['Contig'].isin(contigs)].reset_index()

In [ ]:
labels = tb['Contig'].unique()
input_dropdown = alt.binding_select(options=labels, name='Contig: ')
selection = alt.selection_point("chrom_choice", fields=['Contig'], value=labels[0], bind=input_dropdown)
highlight = alt.selection_point(name="highlight", on="pointerover", empty=False)
stroke_color = (
    alt.when(highlight).then(alt.value("#7ae00d"))
    .otherwise(alt.value("transparent"))
)
tb['Genomic Interval (bp)'] = [f"{i}-{j}" for i,j in zip(tb['Position'], tb['Position End'])]
tb = tb.rename(columns = {"depth" : "Read Depth", "mol_depth" : "Molecule Depth"})

if len(tb) < 15000:
    alt.data_transformers.disable_max_rows()
    (
        alt.Chart(tb)        
        .mark_bar(strokeWidth=1.5)
        .encode(
            x=alt.X('Position:Q')
                .scale(domainMin=0)
                .axis(alt.Axis(title='Position (Mb)', labelExpr='datum.value / 1000000')),
            y = alt.Y('Depth:Q', title = None),
            color = alt.Color('Metric:N').legend(None),
            stroke = stroke_color,
            row = "Metric:N",
            tooltip = [
                "Metric:N",
                'Genomic Interval (bp):N',
                'Depth:Q'
            ]
        )
        .resolve_scale(x='independent', y='independent')
        .transform_filter(selection)
        .transform_fold(['Read Depth', 'Molecule Depth'], ['Metric', 'Depth'])
        .add_params(selection, highlight)
        .properties(
            title = "Depth Across the Genome",
            height = 150, width = 650,
            usermeta={'embedOptions': {'downloadFileName': f'{samplename}.depth'}}
        )
    ).interactive().display()
else:
    print_html("Unable to plot due to the dataframe being too large for Altair/Vega-lite to process")

## Supporting info
:::{dropdown} Interpreting the input file
Below are the descriptions of the columns in this sample's associated `.bxstats.gz` file, which was created by Harpy using the included `bx_stats` script. The term `molecule` refers to the `MI:i` tag in the alignments, which is a unique molecule ID given to the original fragment alignments sharing a barcode are inferred to have originated from. The inference takes into account an [alignment distance threshold](https://pdimens.github.io/harpy/haplotagdata/#barcode-thresholds) and that the sequences aligned to the same contig.

**contig**: name of the contig the molecule occurs on

**molecule**: the molecule name as given by the MI:i: tag

**reads**: number of alignments associated with this molecule

**start**: the start position of the first alignment for that molecule

**end**: the end position of the last alignment for that molecule

**length_inferred**: inferred length of the molecule based on the start/end of the alignments sharing the same barcode

**percent_coverage**: what percent of the molecule is represented by sequence alignments

**aligned_bp**: total number of base pairs aligned for that molecule
:::

:::{dropdown} Linked-Read terminology
```{card} Barcode Status
BX barcode validity is classified into one of three categories:

**Valid**: A complete BX barcode was present in the read (i.e. no 00 for any segments)

**Invalid**: A barcode was present in the read, but it contained `00` in at least one of the barcode segments (haplotagging), `0` as one of the segments (stlfr), or an `N` (tellseq/10x).

**Missing**: There is no barcode in the read. For technical reasons this is usually equivalent to `invalid`
```

```{card} Molecules
Linked-read data is specific for the definition of a "molecule":

**Unique/Inferred Molecules**: Given linked-read barcode information, the original piece of DNA from which the sequenced fragments are considered to originate from.

**Inferred Sequence**:  While somewhat similar to "inferred molecule", the inferred sequence describes the original DNA fragment that was put on the sequencer. If the fragment was longer than the sequencer could fully sequence, e.g. 400bp fragment and the sequencer can only sequence 300bp, then the inferred sequence is 400bp long, even though only 300bp are represented in the sequence data. If the entire fragment was sequenced, then the inferred length and sequence lengths should be identical.
```

```{card} Coverage and Depth
There are several kinds of "coverage" when working with linked-read data:

**Aligned Depth/Coverage**: The standard interpretation of depth comparing the number of aligned base-pairs to the genome or contigs

**Molecule Coverage**: The coverage breadth or depth of sequences onto *unique molecules* (rather than the genome), as inferred from linked-read barcodes

**Linked Depth/Coverage**: The coverage breadth or depth that *includes unsequenced gaps between linked sequences* that are associated with a single unique molecule. For example, if two 300bp paired-end reads share the same barcode and map 2000bp apart, the calculation *includes* the 1400bp between the sequences as if they were present. This is similar to *inferred sequences* described above, except spanning across linked sequences rather than within a paired-end sequence.
```
:::